In [1]:
import torch
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'
device

'cuda'

In [2]:
import torch
import torch.nn as nn

layer = nn.Linear(40, 10)
layer.weight.data *= 6 ** 0.5
torch.zero_(layer.bias.data)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [3]:
layer = nn.Linear(40, 10)
nn.init.kaiming_uniform_(layer.weight)
nn.init.zeros_(layer.bias)

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)

In [4]:
def use_he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight)
        nn.init.zeros_(module.bias)
model = nn.Sequential(nn.Linear(50, 40), nn.ReLU(), nn.Linear(40, 1),nn.ReLU())
model.apply(use_he_init)

Sequential(
  (0): Linear(in_features=50, out_features=40, bias=True)
  (1): ReLU()
  (2): Linear(in_features=40, out_features=1, bias=True)
  (3): ReLU()
)

In [5]:
model = nn.Sequential(nn.Linear(50,40), nn.LeakyReLU(negative_slope=0.2))
model.apply(use_he_init)

Sequential(
  (0): Linear(in_features=50, out_features=40, bias=True)
  (1): LeakyReLU(negative_slope=0.2)
)

In [6]:
torch.manual_seed(42)
alpha = 0.2
model = nn.Sequential(nn.Linear(50,40), nn.LeakyReLU(negative_slope=alpha))
nn.init.kaiming_uniform_(model[0].weight,alpha, nonlinearity='leaky_relu')
model(torch.rand(2,50)).shape

torch.Size([2, 40])

In [7]:
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(50, 40), nn.ELU())
nn.init.kaiming_uniform_(model[0].weight)
model(torch.rand(2,50)).shape

torch.Size([2, 40])

In [8]:
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(50,40), nn.SELU())
nn.init.kaiming_uniform_(model[0].weight)
model(torch.rand(2,50)).shape

torch.Size([2, 40])

In [9]:
class SwiGLU(nn.Module):
    def __init__(self, beta=1.0):
        super().__init__()
        self.beta = beta
    
    def forward(self, x):
        z1, z2 = x.chunk(2, dim=-1)
        param_swish = z1 * torch.sigmoid(self.beta * z1)
        return param_swish * z2
    
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(50, 2 * 40), SwiGLU(beta=0.2))
nn.init.kaiming_uniform_(model[0].weight)
model(torch.rand(2,50)).shape

torch.Size([2, 40])

In [10]:
import torch.nn.functional as F

class ReLU2(nn.Module):
    def forward(self, x):
        return F.relu(x).square()

torch.manual_seed(42)
model = nn.Sequential(nn.Linear(50, 40), ReLU2())
nn.init.kaiming_uniform_(model[0].weight)
model(torch.rand(2,50)).shape

torch.Size([2, 40])

In [11]:
torch.manual_seed(42)

model = nn.Sequential(
    nn.Flatten(),
    nn.BatchNorm1d(1 * 28 * 28),
    nn.Linear(1 * 28 * 28, 300),
    nn.ReLU(),
    nn.BatchNorm1d(300),
    nn.Linear(300,100),
    nn.BatchNorm1d(100),
    nn.Linear(100,10)
)

In [12]:
dict(model[1].named_parameters()).keys()

dict_keys(['weight', 'bias'])

In [13]:
dict(model[1].named_buffers()).keys()

dict_keys(['running_mean', 'running_var', 'num_batches_tracked'])

In [14]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(1 * 28 * 28, 300, bias=False),
    nn.BatchNorm1d(300),
    nn.ReLU(),
    nn.Linear(300,100, bias=False),
    nn.BatchNorm1d(100),
    nn.ReLU(),
    nn.Linear(100, 10)
)

In [15]:
torch.manual_seed(42)

inputs = torch.randn(32, 3, 100, 200)
layer_norm = nn.LayerNorm([100, 200])
result = layer_norm(inputs)

In [16]:
means = inputs.mean(dim=[2, 3], keepdim=True)
vars_ = inputs.var(dim=[2, 3], keepdim=True, unbiased=False)
stds = torch.sqrt(vars_ + layer_norm.eps)
result2 = layer_norm.weight * (inputs - means) / stds + layer_norm.bias
assert torch.allclose(result, result2)

In [17]:
layer_norm = nn.LayerNorm([3, 100, 200])
result = layer_norm(inputs)

In [19]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_openml

fashion_mnist = fetch_openml(name='Fashion-MNIST', as_frame=False)
X = torch.FloatTensor(fashion_mnist.data.reshape(-1,1,28,28)/255.)
y = torch.from_numpy(fashion_mnist.target.astype(int))
in_B = (y==0) | (y==2)
X_A, y_A = X[~in_B], y[~in_B]
y_A = torch.maximum(y_A - 2, torch.tensor(0))
X_B, y_B = X[in_B], (y[in_B] == 2).to(dtype=torch.float32).view(-1,1)

train_set_A = TensorDataset(X_A[:-7_000],y_A[:-7_000])
valid_set_A = TensorDataset(X_A[-7_000:-5000], y_A[-7000:-5000])
test_set_A = TensorDataset(X_A[-5_000:], y_A[-5000:])
train_set_B = TensorDataset(X_B[:20], y_B[:20])
valid_set_B = TensorDataset(X_B[20:5000], y_B[20:5000])
test_set_B = TensorDataset(X_B[5_000:],y_B[5000:])

train_loader_A = DataLoader(train_set_A, batch_size=32, shuffle=True)
valid_loader_A = DataLoader(valid_set_A, batch_size=32)
test_loader_A = DataLoader(test_set_A, batch_size=32)
train_loader_B = DataLoader(train_set_B, batch_size=32, shuffle=True)
valid_loader_B = DataLoader(valid_set_B, batch_size=32)
test_loader_B = DataLoader(test_set_B, batch_size=32)

In [21]:
%pip install torchmetrics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 62.7 MB/s eta 0:00:00


In [23]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device),y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  #Gradient Clipping
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [24]:
torch.manual_seed(42)

model_A = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 8)
)
model_A = model_A.to(device)

In [25]:
model_A.apply(use_he_init)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=100, bias=True)
  (2): ReLU()
  (3): Linear(in_features=100, out_features=100, bias=True)
  (4): ReLU()
  (5): Linear(in_features=100, out_features=100, bias=True)
  (6): ReLU()
  (7): Linear(in_features=100, out_features=8, bias=True)
)

In [26]:
n_epochs = 20
optimizer = torch.optim.SGD(model_A.parameters(), lr=0.005)
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=8).to(device)
history_A = train(model_A, optimizer, xentropy, accuracy,
                  train_loader_A, valid_loader_A, n_epochs)

Epoch 1/20, train loss: 0.8637, train metric: 0.7115, valid metric: 0.8265
Epoch 2/20, train loss: 0.4608, train metric: 0.8461, valid metric: 0.8570
Epoch 3/20, train loss: 0.3836, train metric: 0.8699, valid metric: 0.8730
Epoch 4/20, train loss: 0.3465, train metric: 0.8820, valid metric: 0.8825
Epoch 5/20, train loss: 0.3236, train metric: 0.8884, valid metric: 0.8935
Epoch 6/20, train loss: 0.3080, train metric: 0.8933, valid metric: 0.8935
Epoch 7/20, train loss: 0.2958, train metric: 0.8977, valid metric: 0.8910
Epoch 8/20, train loss: 0.2861, train metric: 0.9007, valid metric: 0.9015
Epoch 9/20, train loss: 0.2784, train metric: 0.9036, valid metric: 0.8980
Epoch 10/20, train loss: 0.2708, train metric: 0.9068, valid metric: 0.8950
Epoch 11/20, train loss: 0.2658, train metric: 0.9085, valid metric: 0.9045
Epoch 12/20, train loss: 0.2595, train metric: 0.9095, valid metric: 0.8990
Epoch 13/20, train loss: 0.2548, train metric: 0.9120, valid metric: 0.9080
Epoch 14/20, train lo

In [28]:
torch.manual_seed(9)

model_B = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100,100),
    nn.ReLU(),
    nn.Linear(100,1)
).to(device)

In [29]:
model_B.apply(use_he_init)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=100, bias=True)
  (2): ReLU()
  (3): Linear(in_features=100, out_features=100, bias=True)
  (4): ReLU()
  (5): Linear(in_features=100, out_features=100, bias=True)
  (6): ReLU()
  (7): Linear(in_features=100, out_features=1, bias=True)
)

In [30]:
n_epochs = 20
optimizer = torch.optim.SGD(model_B.parameters(), lr=0.005)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task='binary').to(device)
history_B = train(model_B, optimizer, xentropy, accuracy,
                  train_loader_B, valid_loader_B, n_epochs)

Epoch 1/20, train loss: 1.1117, train metric: 0.4000, valid metric: 0.5030
Epoch 2/20, train loss: 1.0771, train metric: 0.4000, valid metric: 0.5030
Epoch 3/20, train loss: 1.0432, train metric: 0.4000, valid metric: 0.5030
Epoch 4/20, train loss: 1.0097, train metric: 0.4000, valid metric: 0.5030
Epoch 5/20, train loss: 0.9777, train metric: 0.4000, valid metric: 0.5030
Epoch 6/20, train loss: 0.9464, train metric: 0.4000, valid metric: 0.5032
Epoch 7/20, train loss: 0.9156, train metric: 0.4000, valid metric: 0.5032
Epoch 8/20, train loss: 0.8864, train metric: 0.4000, valid metric: 0.5032
Epoch 9/20, train loss: 0.8586, train metric: 0.4000, valid metric: 0.5032
Epoch 10/20, train loss: 0.8321, train metric: 0.4000, valid metric: 0.5030
Epoch 11/20, train loss: 0.8067, train metric: 0.4000, valid metric: 0.5032
Epoch 12/20, train loss: 0.7824, train metric: 0.4000, valid metric: 0.5034
Epoch 13/20, train loss: 0.7593, train metric: 0.4000, valid metric: 0.5038
Epoch 14/20, train lo

In [31]:
evaluate_tm(model_B, test_loader_B, accuracy)

tensor(0.5656, device='cuda:0')

In [32]:
import copy

torch.manual_seed(43)
reused_layers = copy.deepcopy(model_A[:-1])
model_B_on_A = nn.Sequential(
    *reused_layers,
    nn.Linear(100,1)
).to(device)

In [33]:
for layer in model_B_on_A[:-1]:
    for param in layer.parameters():
        param.requires_grad = False

In [34]:
n_epochs = 10
optimizer = torch.optim.SGD(model_B_on_A.parameters(), lr=0.005)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task='binary').to(device)
history_B = train(model_B_on_A, optimizer, xentropy, accuracy,
                  train_loader_B, valid_loader_B, n_epochs)

Epoch 1/10, train loss: 0.8792, train metric: 0.4000, valid metric: 0.5038
Epoch 2/10, train loss: 0.8450, train metric: 0.4000, valid metric: 0.5006
Epoch 3/10, train loss: 0.8146, train metric: 0.4000, valid metric: 0.4988
Epoch 4/10, train loss: 0.7880, train metric: 0.4000, valid metric: 0.4894
Epoch 5/10, train loss: 0.7653, train metric: 0.3500, valid metric: 0.4797
Epoch 6/10, train loss: 0.7464, train metric: 0.3000, valid metric: 0.4835
Epoch 7/10, train loss: 0.7308, train metric: 0.3500, valid metric: 0.5165
Epoch 8/10, train loss: 0.7180, train metric: 0.4000, valid metric: 0.5502
Epoch 9/10, train loss: 0.7069, train metric: 0.6000, valid metric: 0.5737
Epoch 10/10, train loss: 0.6968, train metric: 0.6000, valid metric: 0.5904


In [40]:
for layer in model_B_on_A[2:]:
    for param in layer.parameters():
        param.requires_grad = True

In [41]:
n_epochs=20
optimizer = torch.optim.SGD(model_B_on_A.parameters(), lr=0.005)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task='binary').to(device)
history_B = train(model_B_on_A, optimizer, xentropy, accuracy,
                  train_loader_B, valid_loader_B, n_epochs)

Epoch 1/20, train loss: 0.6871, train metric: 0.6000, valid metric: 0.6137
Epoch 2/20, train loss: 0.6765, train metric: 0.6000, valid metric: 0.6309
Epoch 3/20, train loss: 0.6661, train metric: 0.6000, valid metric: 0.6552
Epoch 4/20, train loss: 0.6559, train metric: 0.7000, valid metric: 0.6733
Epoch 5/20, train loss: 0.6457, train metric: 0.7000, valid metric: 0.6922
Epoch 6/20, train loss: 0.6357, train metric: 0.7500, valid metric: 0.7094
Epoch 7/20, train loss: 0.6259, train metric: 0.7500, valid metric: 0.7293
Epoch 8/20, train loss: 0.6161, train metric: 0.7500, valid metric: 0.7470
Epoch 9/20, train loss: 0.6065, train metric: 0.7500, valid metric: 0.7643
Epoch 10/20, train loss: 0.5971, train metric: 0.7500, valid metric: 0.7797
Epoch 11/20, train loss: 0.5878, train metric: 0.8500, valid metric: 0.7920
Epoch 12/20, train loss: 0.5786, train metric: 0.8500, valid metric: 0.8052
Epoch 13/20, train loss: 0.5695, train metric: 0.8500, valid metric: 0.8205
Epoch 14/20, train lo

In [42]:
evaluate_tm(model_B_on_A, test_loader_B, accuracy)

tensor(0.8926, device='cuda:0')

In [44]:
train_set = TensorDataset(X[:55_000], y[:55_000])
valid_set = TensorDataset(X[55_000:60_000], y[55_000:60_000])
test_set = TensorDataset(X[60_000:], y[60_000:])

def build_model(seed=42):
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, 100),
        nn.ReLU(),
        nn.Linear(100, 100),
        nn.ReLU(),
        nn.Linear(100, 100),
        nn.ReLU(),
        nn.Linear(100, 10)
    ).to(device)
    model.apply(use_he_init)
    return model

def test_optimizer(model, optimizer, n_epochs=10, batch_size=32):
    train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
    valid_loader = DataLoader(valid_set, batch_size=32)
    test_loader = DataLoader(test_set, batch_size=32)
    xentropy = nn.CrossEntropyLoss()
    accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10)
    history = train(model, optimizer, xentropy, accuracy.to(device),
                    train_loader, valid_loader, n_epochs)
    return history, evaluate_tm(model, test_loader, accuracy)

In [45]:
model = build_model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
history_sgd, acc_sgd = test_optimizer(model, optimizer)

Epoch 1/10, train loss: 0.6426, train metric: 0.7748, valid metric: 0.8388
Epoch 2/10, train loss: 0.4562, train metric: 0.8388, valid metric: 0.8502
Epoch 3/10, train loss: 0.4068, train metric: 0.8555, valid metric: 0.8536
Epoch 4/10, train loss: 0.3788, train metric: 0.8650, valid metric: 0.8636
Epoch 5/10, train loss: 0.3575, train metric: 0.8717, valid metric: 0.8660
Epoch 6/10, train loss: 0.3415, train metric: 0.8780, valid metric: 0.8740
Epoch 7/10, train loss: 0.3275, train metric: 0.8814, valid metric: 0.8746
Epoch 8/10, train loss: 0.3159, train metric: 0.8859, valid metric: 0.8724
Epoch 9/10, train loss: 0.3065, train metric: 0.8883, valid metric: 0.8724
Epoch 10/10, train loss: 0.2967, train metric: 0.8924, valid metric: 0.8806


In [47]:
model = build_model()
optimizer = torch.optim.SGD(model.parameters(), momentum=0.9, lr=0.05)
history_momentum, acc_momentum = test_optimizer(model, optimizer)

Epoch 1/10, train loss: 0.5693, train metric: 0.7964, valid metric: 0.8044
Epoch 2/10, train loss: 0.4460, train metric: 0.8413, valid metric: 0.8540
Epoch 3/10, train loss: 0.4037, train metric: 0.8567, valid metric: 0.8510
Epoch 4/10, train loss: 0.3824, train metric: 0.8641, valid metric: 0.8586
Epoch 5/10, train loss: 0.3618, train metric: 0.8704, valid metric: 0.8554
Epoch 6/10, train loss: 0.3532, train metric: 0.8735, valid metric: 0.8698
Epoch 7/10, train loss: 0.3428, train metric: 0.8781, valid metric: 0.8618
Epoch 8/10, train loss: 0.3316, train metric: 0.8822, valid metric: 0.8704
Epoch 9/10, train loss: 0.3270, train metric: 0.8833, valid metric: 0.8790
Epoch 10/10, train loss: 0.3160, train metric: 0.8858, valid metric: 0.8714


In [48]:
model = build_model()
optimizer = torch.optim.SGD(model.parameters(),
                            momentum=0.9, nesterov=True, lr=0.05)
history_nesterov, acc_nesterov = test_optimizer(model, optimizer)

Epoch 1/10, train loss: 0.5321, train metric: 0.8085, valid metric: 0.8316
Epoch 2/10, train loss: 0.4275, train metric: 0.8481, valid metric: 0.8500
Epoch 3/10, train loss: 0.3943, train metric: 0.8596, valid metric: 0.8516
Epoch 4/10, train loss: 0.3696, train metric: 0.8666, valid metric: 0.8612
Epoch 5/10, train loss: 0.3525, train metric: 0.8734, valid metric: 0.8646
Epoch 6/10, train loss: 0.3448, train metric: 0.8773, valid metric: 0.8632
Epoch 7/10, train loss: 0.3329, train metric: 0.8790, valid metric: 0.8730
Epoch 8/10, train loss: 0.3264, train metric: 0.8837, valid metric: 0.8686
Epoch 9/10, train loss: 0.3174, train metric: 0.8868, valid metric: 0.8684
Epoch 10/10, train loss: 0.3085, train metric: 0.8887, valid metric: 0.8702


In [49]:
model = build_model()
optimizer = torch.optim.Adagrad(model.parameters(), lr=0.05)
history_adagrad, acc_adagrad = test_optimizer(model, optimizer)

Epoch 1/10, train loss: 0.5986, train metric: 0.8056, valid metric: 0.8564
Epoch 2/10, train loss: 0.3710, train metric: 0.8634, valid metric: 0.8660
Epoch 3/10, train loss: 0.3368, train metric: 0.8766, valid metric: 0.8660
Epoch 4/10, train loss: 0.3157, train metric: 0.8839, valid metric: 0.8730
Epoch 5/10, train loss: 0.3003, train metric: 0.8888, valid metric: 0.8778
Epoch 6/10, train loss: 0.2878, train metric: 0.8939, valid metric: 0.8814
Epoch 7/10, train loss: 0.2761, train metric: 0.8980, valid metric: 0.8826
Epoch 8/10, train loss: 0.2686, train metric: 0.9010, valid metric: 0.8790
Epoch 9/10, train loss: 0.2600, train metric: 0.9032, valid metric: 0.8808
Epoch 10/10, train loss: 0.2532, train metric: 0.9059, valid metric: 0.8844


In [50]:
model = build_model()
optimizer = torch.optim.RMSprop(model.parameters(), alpha=0.9, lr=0.05)
history_rmsprop, acc_rmsprop = test_optimizer(model, optimizer)

Epoch 1/10, train loss: 3.3600, train metric: 0.4728, valid metric: 0.5102
Epoch 2/10, train loss: 1.3634, train metric: 0.5180, valid metric: 0.5222
Epoch 3/10, train loss: 1.4490, train metric: 0.4768, valid metric: 0.4378
Epoch 4/10, train loss: 1.5632, train metric: 0.4163, valid metric: 0.4020
Epoch 5/10, train loss: 1.6493, train metric: 0.3994, valid metric: 0.4460
Epoch 6/10, train loss: 1.6638, train metric: 0.4325, valid metric: 0.2134
Epoch 7/10, train loss: 1.9073, train metric: 0.2924, valid metric: 0.3394
Epoch 8/10, train loss: 1.7177, train metric: 0.3964, valid metric: 0.4204
Epoch 9/10, train loss: 1.5670, train metric: 0.4534, valid metric: 0.4568
Epoch 10/10, train loss: 1.5247, train metric: 0.4398, valid metric: 0.4588


In [51]:
model = build_model()
optimizer = torch.optim.Adam(model.parameters(), betas=(0.9, 0.999), lr=0.05)
history_adam, acc_adam = test_optimizer(model, optimizer)

Epoch 1/10, train loss: 1.9298, train metric: 0.3197, valid metric: 0.3518
Epoch 2/10, train loss: 1.5543, train metric: 0.3756, valid metric: 0.4338
Epoch 3/10, train loss: 1.4543, train metric: 0.4198, valid metric: 0.4288
Epoch 4/10, train loss: 1.6037, train metric: 0.3923, valid metric: 0.3774
Epoch 5/10, train loss: 1.7300, train metric: 0.3610, valid metric: 0.3574
Epoch 6/10, train loss: 1.5587, train metric: 0.3587, valid metric: 0.3604
Epoch 7/10, train loss: 1.5542, train metric: 0.3571, valid metric: 0.3282
Epoch 8/10, train loss: 1.5540, train metric: 0.3592, valid metric: 0.3634
Epoch 9/10, train loss: 1.5953, train metric: 0.3627, valid metric: 0.3694
Epoch 10/10, train loss: 1.4644, train metric: 0.3842, valid metric: 0.4166


In [55]:
model = build_model()
optimizer = torch.optim.Adamax(model.parameters(), betas=(0.9,0.999), lr=0.05)
history_adamax, acc_adamax = test_optimizer(model, optimizer)

Epoch 1/10, train loss: 0.7200, train metric: 0.7861, valid metric: 0.8216
Epoch 2/10, train loss: 0.4670, train metric: 0.8387, valid metric: 0.8406
Epoch 3/10, train loss: 0.4351, train metric: 0.8497, valid metric: 0.8390
Epoch 4/10, train loss: 0.4163, train metric: 0.8536, valid metric: 0.8462
Epoch 5/10, train loss: 0.4000, train metric: 0.8597, valid metric: 0.8552
Epoch 6/10, train loss: 0.3917, train metric: 0.8637, valid metric: 0.8640
Epoch 7/10, train loss: 0.3848, train metric: 0.8652, valid metric: 0.8618
Epoch 8/10, train loss: 0.3707, train metric: 0.8680, valid metric: 0.8564
Epoch 9/10, train loss: 0.3686, train metric: 0.8716, valid metric: 0.8606
Epoch 10/10, train loss: 0.3633, train metric: 0.8737, valid metric: 0.8628


In [56]:
model = build_model()
optimizer = torch.optim.NAdam(model.parameters(), betas=(0.9,0.999),lr=0.05)
history_nadam, acc_nadam = test_optimizer(model, optimizer)

Epoch 1/10, train loss: 1.7018, train metric: 0.4528, valid metric: 0.5372
Epoch 2/10, train loss: 1.1792, train metric: 0.5361, valid metric: 0.5344
Epoch 3/10, train loss: 1.1356, train metric: 0.5313, valid metric: 0.5862
Epoch 4/10, train loss: 1.1128, train metric: 0.5514, valid metric: 0.5994
Epoch 5/10, train loss: 1.1135, train metric: 0.5518, valid metric: 0.5318
Epoch 6/10, train loss: 1.2042, train metric: 0.5283, valid metric: 0.5600
Epoch 7/10, train loss: 1.2372, train metric: 0.4691, valid metric: 0.4434
Epoch 8/10, train loss: 1.4372, train metric: 0.3566, valid metric: 0.3390
Epoch 9/10, train loss: 1.4372, train metric: 0.3517, valid metric: 0.3468
Epoch 10/10, train loss: 1.4599, train metric: 0.3546, valid metric: 0.3484


In [57]:
model = build_model()
optimizer = torch.optim.AdamW(model.parameters(), betas=(0.9, 0.999),
                              weight_decay=1e-5, lr=0.05)
history_adamw, acc_adamw = test_optimizer(model, optimizer)

Epoch 1/10, train loss: 1.6255, train metric: 0.4058, valid metric: 0.3438
Epoch 2/10, train loss: 1.4949, train metric: 0.3856, valid metric: 0.3950
Epoch 3/10, train loss: 1.4193, train metric: 0.4261, valid metric: 0.4604
Epoch 4/10, train loss: 1.4956, train metric: 0.4297, valid metric: 0.4340
Epoch 5/10, train loss: 1.4392, train metric: 0.4066, valid metric: 0.3630
Epoch 6/10, train loss: 1.5358, train metric: 0.3639, valid metric: 0.3728
Epoch 7/10, train loss: 1.5097, train metric: 0.3702, valid metric: 0.3620
Epoch 8/10, train loss: 1.5359, train metric: 0.3660, valid metric: 0.3654
Epoch 9/10, train loss: 1.7706, train metric: 0.3662, valid metric: 0.3656
Epoch 10/10, train loss: 1.9718, train metric: 0.3707, valid metric: 0.3966
